# M0000545 Hamilton Cycle 全样本与分段区制切换

本 notebook 使用 `output_宏观_pkl/M0000545.pkl`，对 `中国:工业增加值` 进行 Hamilton cycle 过滤，并分别运行全样本与分段递推的两区制切换模型。结果会导出到 `M0000545_hamcycle_full_segment_output`。



## 1. 参数与输出目录

这一段只做环境和参数设置：指定数据目录、输出目录、指标代码、Hamilton 滤波参数、分段长度以及区制切换模型的敏感度参数。后面如果要做敏感性测试，优先改这里。



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()
PKL_DIR = ROOT / "output_宏观_pkl"
if not PKL_DIR.exists():
    PKL_DIR = ROOT / "ouput_宏观_pkl"

OUT = ROOT / "M0000545_hamcycle_full_segment_output"
OUT.mkdir(exist_ok=True)

START_DATE = pd.Timestamp("2026-05-31")
SERIES_CODE = "M0000545"
SERIES_NAME = "中国:工业增加值"

SEGMENT_MONTHS = 60
MIN_SEGMENT_OBS = 24

HAMILTON_H = 8
HAMILTON_P = 4

TRANSITION_STAY_INIT = 0.80
MIN_SWITCH_PROB = 0.08
LIKELIHOOD_SIGMA_SCALE = 0.85
STATE_PROB_THRESHOLD = 0.50
MAX_ITER = 500
TOL = 1e-8






## 2. 读取月度数据并构造 Hamilton Cycle

这里定义两个数据处理函数：

- `_first_numeric_series`：从 pkl 里抽取第一列可用数值序列。
- `read_monthly_pkl`：统一成月末时间索引，并对少量缺口插值。
- `hamilton_filter`：用 Hamilton 回归方法提取周期项。



In [2]:
def _first_numeric_series(obj) -> pd.Series:
    if isinstance(obj, pd.Series):
        return pd.to_numeric(obj, errors="coerce")
    if isinstance(obj, pd.DataFrame):
        numeric = obj.apply(pd.to_numeric, errors="coerce")
        cols = [col for col in numeric.columns if numeric[col].notna().any()]
        if not cols:
            raise ValueError("DataFrame 中没有可转换成数值的列")
        return numeric[cols[0]]
    return pd.to_numeric(pd.Series(obj), errors="coerce")


def read_monthly_pkl(code: str, start_date: pd.Timestamp = START_DATE) -> pd.Series:
    path = PKL_DIR / f"{code}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    raw = _first_numeric_series(pd.read_pickle(path)).dropna()
    idx_text = pd.Series(raw.index.astype(str)).str.strip()
    parsed_idx = pd.to_datetime(idx_text, errors="coerce")

    if parsed_idx.notna().any():
        s = pd.Series(raw.to_numpy(), index=parsed_idx)
        s = s[s.index.notna()]
    else:
        dates = [start_date - pd.offsets.MonthEnd(i) for i in range(len(raw))]
        s = pd.Series(raw.to_numpy(), index=pd.DatetimeIndex(dates))

    month_end = pd.DatetimeIndex(s.index).to_period("M").to_timestamp("M")
    monthly = pd.Series(s.to_numpy(), index=month_end).sort_index().groupby(level=0).last()
    regular = monthly.reindex(pd.date_range(monthly.index.min(), monthly.index.max(), freq="ME"))
    return regular.interpolate(limit=2).dropna().rename(code)


def hamilton_filter(y: pd.Series, h: int = HAMILTON_H, p: int = HAMILTON_P) -> tuple[pd.Series, pd.Series]:
    x = y.astype(float).dropna()
    rows = []
    targets = []
    dates = []

    for t in range(p - 1, len(x) - h):
        rows.append([1.0] + [x.iloc[t - lag] for lag in range(p)])
        targets.append(x.iloc[t + h])
        dates.append(x.index[t + h])

    if len(rows) <= p:
        raise ValueError("Hamilton 回归样本不足")

    xmat = np.asarray(rows, dtype=float)
    yvec = np.asarray(targets, dtype=float)
    beta = np.linalg.lstsq(xmat, yvec, rcond=None)[0]
    trend = pd.Series(xmat @ beta, index=pd.DatetimeIndex(dates), name="hamilton_trend")
    cycle = (x.reindex(trend.index) - trend).rename("hamilton_cycle")
    return cycle, trend






## 3. 区制切换模型的基础工具

这一段是两状态模型的底层工具：正态密度、转移矩阵约束、初始参数整理、前向滤波，以及把概率结果整理成表。这里一般不用改。



In [3]:
def _normal_pdf(values: np.ndarray, mu: np.ndarray, sigma: float) -> np.ndarray:
    sigma = max(float(sigma), 1e-8)
    z = (values[:, None] - mu[None, :]) / sigma
    return np.exp(-0.5 * z * z) / (np.sqrt(2.0 * np.pi) * sigma)


def _constrain_transition(transition: np.ndarray, min_switch_prob: float) -> np.ndarray:
    transition = np.asarray(transition, dtype=float).copy()
    min_switch_prob = float(np.clip(min_switch_prob, 0.0, 0.49))
    max_stay_prob = 1.0 - min_switch_prob

    for i in range(transition.shape[0]):
        transition[i, i] = min(transition[i, i], max_stay_prob)
        off_diag = [j for j in range(transition.shape[1]) if j != i]
        remaining = 1.0 - transition[i, i]
        off_sum = transition[i, off_diag].sum()
        if off_sum <= 0:
            transition[i, off_diag] = remaining / len(off_diag)
        else:
            transition[i, off_diag] = transition[i, off_diag] / off_sum * remaining

    return transition / transition.sum(axis=1, keepdims=True)


def _params_to_arrays(initial_params: dict | None, values: np.ndarray) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    if initial_params is None:
        q25, q75 = np.nanpercentile(values, [25, 75])
        mu = np.asarray([q25, q75], dtype=float)
        sigma = float(np.nanstd(values, ddof=1)) or 1.0
        stay = float(np.clip(TRANSITION_STAY_INIT, 0.51, 0.99))
        transition = np.asarray([[stay, 1.0 - stay], [1.0 - stay, stay]], dtype=float)
        init_prob = np.asarray([0.50, 0.50], dtype=float)
        return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob

    mu = np.asarray([initial_params["mu_low"], initial_params["mu_high"]], dtype=float)
    sigma = float(initial_params.get("sigma", np.nanstd(values, ddof=1) or 1.0))
    transition = np.asarray(
        [
            [initial_params["p_stay_low"], initial_params["p_switch_low_to_high"]],
            [initial_params["p_switch_high_to_low"], initial_params["p_stay_high"]],
        ],
        dtype=float,
    )
    init_prob = np.asarray(initial_params.get("next_init_prob", [0.50, 0.50]), dtype=float)
    init_prob = init_prob / init_prob.sum()
    return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob


def _forward_filter(
    values: np.ndarray,
    mu: np.ndarray,
    sigma: float,
    transition: np.ndarray,
    init_prob: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float]:
    emission = _normal_pdf(values, mu, sigma)
    prior = np.zeros((len(values), 2))
    filtered = np.zeros((len(values), 2))
    scale = np.zeros(len(values))

    prior[0] = init_prob
    filtered[0] = prior[0] * emission[0]
    scale[0] = max(filtered[0].sum(), 1e-300)
    filtered[0] /= scale[0]

    for t in range(1, len(values)):
        prior[t] = filtered[t - 1] @ transition
        filtered[t] = prior[t] * emission[t]
        scale[t] = max(filtered[t].sum(), 1e-300)
        filtered[t] /= scale[t]

    return prior, filtered, float(np.log(scale).sum())


def _state_frame(index: pd.DatetimeIndex, values: np.ndarray, prior: np.ndarray, filtered: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(index=index)
    out["model_value"] = values
    out["prior_prob_low_state"] = prior[:, 0]
    out["prior_prob_high_state"] = prior[:, 1]
    out["prob_low_state"] = filtered[:, 0]
    out["prob_high_state"] = filtered[:, 1]
    out["state"] = np.where(out["prob_high_state"] >= STATE_PROB_THRESHOLD, "高均值状态", "低均值状态")
    out["regime_label"] = np.where(
        out["prob_high_state"] >= STATE_PROB_THRESHOLD,
        "工业增加值高位区制",
        "工业增加值低位区制",
    )
    return out






## 4. 拟合与递推函数

这里定义正式使用的模型函数：

- `fit_hmm`：对当前样本估计两区制模型。
- `apply_params_to_segment`：用上一段参数直接测算下一段，用来做跨段递推检验。
- `make_segments`：按固定月数切分 Hamilton cycle。



In [4]:
def fit_hmm(y: pd.Series, initial_params: dict | None = None) -> tuple[pd.DataFrame, dict]:
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    if len(values) < MIN_SEGMENT_OBS:
        raise ValueError(f"样本不足：{len(values)} < {MIN_SEGMENT_OBS}")

    mu, sigma, transition, init_prob = _params_to_arrays(initial_params, values)
    last_ll = -np.inf
    ll = np.nan

    for iteration in range(1, MAX_ITER + 1):
        emission = _normal_pdf(values, mu, sigma)
        alpha = np.zeros((len(values), 2))
        scale = np.zeros(len(values))

        alpha[0] = init_prob * emission[0]
        scale[0] = max(alpha[0].sum(), 1e-300)
        alpha[0] /= scale[0]

        for t in range(1, len(values)):
            alpha[t] = (alpha[t - 1] @ transition) * emission[t]
            scale[t] = max(alpha[t].sum(), 1e-300)
            alpha[t] /= scale[t]

        beta = np.ones((len(values), 2))
        for t in range(len(values) - 2, -1, -1):
            beta[t] = transition @ (emission[t + 1] * beta[t + 1])
            beta[t] /= scale[t + 1]

        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)

        xi_sum = np.zeros((2, 2))
        for t in range(len(values) - 1):
            xi = alpha[t, :, None] * transition * emission[t + 1, None, :] * beta[t + 1, None, :]
            xi_sum += xi / max(xi.sum(), 1e-300)

        init_prob = gamma[0]
        transition = _constrain_transition(xi_sum / xi_sum.sum(axis=1, keepdims=True), MIN_SWITCH_PROB)
        weights = gamma.sum(axis=0)
        mu = (gamma * values[:, None]).sum(axis=0) / weights
        sigma = np.sqrt(((gamma * (values[:, None] - mu[None, :]) ** 2).sum()) / len(values))

        ll = float(np.log(scale).sum())
        if abs(ll - last_ll) < TOL:
            break
        last_ll = ll

    order = np.argsort(mu)
    mu = mu[order]
    transition = transition[np.ix_(order, order)]
    init_prob = init_prob[order]
    init_prob = init_prob / init_prob.sum()

    sigma_for_filter = max(float(sigma) * LIKELIHOOD_SIGMA_SCALE, 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    states = _state_frame(x.index, values, prior, filtered)

    params = {
        "mu_low": float(mu[0]),
        "mu_high": float(mu[1]),
        "sigma": float(sigma),
        "sigma_for_filter": float(sigma_for_filter),
        "p_stay_low": float(transition[0, 0]),
        "p_switch_low_to_high": float(transition[0, 1]),
        "p_switch_high_to_low": float(transition[1, 0]),
        "p_stay_high": float(transition[1, 1]),
        "init_prob_low": float(init_prob[0]),
        "init_prob_high": float(init_prob[1]),
        "next_init_prob": filtered[-1].tolist(),
        "log_likelihood": float(ll),
        "filtered_log_likelihood": float(filtered_ll),
        "iterations": int(iteration),
    }
    return states, params


def apply_params_to_segment(y: pd.Series, params: dict) -> pd.DataFrame:
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    mu, sigma, transition, init_prob = _params_to_arrays(params, values)
    sigma_for_filter = float(params.get("sigma_for_filter", sigma * LIKELIHOOD_SIGMA_SCALE))
    prior, filtered, ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    out = _state_frame(x.index, values, prior, filtered)
    out["log_likelihood_under_previous_params"] = ll
    return out


def make_segments(series: pd.Series, segment_months: int = SEGMENT_MONTHS) -> list[pd.Series]:
    x = series.dropna().sort_index()
    segments = []
    for start in range(0, len(x), segment_months):
        part = x.iloc[start : start + segment_months]
        if len(part) >= MIN_SEGMENT_OBS:
            segments.append(part)
    return segments






## 5. 读取 M0000545 并生成 Hamilton Cycle

运行这一段后，可以先看 `ham_data.tail(12)`，确认原始工业增加值、Hamilton 趋势项和周期项是否正常。



In [5]:
raw_series = read_monthly_pkl(SERIES_CODE)
ham_cycle, ham_trend = hamilton_filter(raw_series)

ham_data = pd.concat(
    {
        "raw_value": raw_series,
        "hamilton_trend": ham_trend,
        "hamilton_cycle": ham_cycle,
    },
    axis=1,
).rename_axis("date")


display(ham_data.tail(12))




,raw_value,hamilton_trend,hamilton_cycle
date,,,
2025-04-30,6.1,6.991780,-0.891780
2025-05-31,5.8,7.005542,-1.205542
2025-06-30,6.8,7.056152,-0.256152
2025-07-31,5.7,7.062592,-1.362592
2025-08-31,5.2,7.283679,-2.083679
2025-09-30,6.5,4.658889,1.841111
2025-10-31,4.9,8.922463,-4.022463
2025-11-30,4.8,10.089295,-5.289295
2025-12-31,5.2,7.795359,-2.595359


## 6. 全样本区制切换模型

这一段把全部 Hamilton cycle 样本一次性放入两状态模型，得到全样本视角下每个月属于高/低均值状态的概率。



In [6]:
full_states, full_params = fit_hmm(ham_cycle)
full_states = full_states.rename_axis("date").reset_index()
full_states.insert(0, "model_scope", "full_sample")
full_states.insert(1, "code", SERIES_CODE)
full_states.insert(2, "name", SERIES_NAME)

full_parameter_table = pd.DataFrame(
    [
        {
            "model_scope": "full_sample",
            "code": SERIES_CODE,
            "name": SERIES_NAME,
            "start": ham_cycle.index.min(),
            "end": ham_cycle.index.max(),
            "nobs": len(ham_cycle),
            **{k: v for k, v in full_params.items() if k != "next_init_prob"},
        }
    ]
)


display(full_parameter_table)
display(full_states.tail(12))




,model_scope,code,name,start,end,nobs,mu_low,mu_high,sigma,sigma_for_filter,p_stay_low,p_switch_low_to_high,p_switch_high_to_low,p_stay_high,init_prob_low,init_prob_high,log_likelihood,filtered_log_likelihood,iterations
0,full_sample,M0000545,中国:工业增加值,2005-12-31,2026-03-31,244,-1.948174,6.320856,5.410018,4.598515,0.92,0.08,0.132743,0.867257,1.039761e-26,1.0,-782.736879,-788.928583,30


,model_scope,code,name,date,model_value,prior_prob_low_state,prior_prob_high_state,prob_low_state,prob_high_state,state,regime_label
232,full_sample,M0000545,中国:工业增加值,2025-04-30,-0.891780,0.317181,0.682819,0.607522,0.392478,低均值状态,工业增加值低位区制
233,full_sample,M0000545,中国:工业增加值,2025-05-31,-1.205542,0.611019,0.388981,0.855444,0.144556,低均值状态,工业增加值低位区制
234,full_sample,M0000545,中国:工业增加值,2025-06-30,-0.256152,0.806197,0.193803,0.915336,0.084664,低均值状态,工业增加值低位区制
235,full_sample,M0000545,中国:工业增加值,2025-07-31,-1.362592,0.853347,0.146653,0.958864,0.041136,低均值状态,工业增加值低位区制
236,full_sample,M0000545,中国:工业增加值,2025-08-31,-2.083679,0.887616,0.112384,0.976714,0.023286,低均值状态,工业增加值低位区制
237,full_sample,M0000545,中国:工业增加值,2025-09-30,1.841111,0.901668,0.098332,0.913006,0.086994,低均值状态,工业增加值低位区制
238,full_sample,M0000545,中国:工业增加值,2025-10-31,-4.022463,0.851513,0.148487,0.984849,0.015151,低均值状态,工业增加值低位区制
239,full_sample,M0000545,中国:工业增加值,2025-11-30,-5.289295,0.908072,0.091928,0.994587,0.005413,低均值状态,工业增加值低位区制
240,full_sample,M0000545,中国:工业增加值,2025-12-31,-2.595359,0.915739,0.084261,0.986014,0.013986,低均值状态,工业增加值低位区制
241,full_sample,M0000545,中国:工业增加值,2026-01-31,-3.677725,0.908990,0.091010,0.989993,0.010007,低均值状态,工业增加值低位区制


## 7. 分段递推区制切换模型

这一段按 `SEGMENT_MONTHS=60` 切分样本。每段重新估计模型，并把本段参数作为下一段初值，同时用上一段参数直接评估下一段，便于观察参数是否稳定。



In [7]:
segments = make_segments(ham_cycle)
current_state_tables = []
next_eval_tables = []
segment_rows = []
parameter_rows = []

previous_params = None
for i, segment in enumerate(segments, start=1):
    segment_id = f"segment_{i:02d}"
    init_source = "sample_quantiles" if previous_params is None else f"segment_{i - 1:02d}_params"
    states, params = fit_hmm(segment, initial_params=previous_params)

    states = states.rename_axis("date").reset_index()
    states.insert(0, "segment_id", segment_id)
    states.insert(1, "evaluation_type", "current_segment_refit")
    current_state_tables.append(states)

    segment_rows.append(
        {
            "segment_id": segment_id,
            "init_source": init_source,
            "start": segment.index.min(),
            "end": segment.index.max(),
            "months": len(segment),
            "latest_date": states["date"].iloc[-1],
            "latest_prob_high_state": states["prob_high_state"].iloc[-1],
            "latest_state": states["state"].iloc[-1],
            "latest_regime_label": states["regime_label"].iloc[-1],
            "state_switches": int(states["state"].ne(states["state"].shift()).sum() - 1),
            "avg_abs_prob_high_change": states["prob_high_state"].diff().abs().mean(),
        }
    )

    parameter_rows.append(
        {
            "segment_id": segment_id,
            "init_source": init_source,
            "start": segment.index.min(),
            "end": segment.index.max(),
            **{k: v for k, v in params.items() if k != "next_init_prob"},
        }
    )

    if i < len(segments):
        next_segment = segments[i]
        next_eval = apply_params_to_segment(next_segment, params)
        next_eval = next_eval.rename_axis("date").reset_index()
        next_eval.insert(0, "source_segment_id", segment_id)
        next_eval.insert(1, "target_segment_id", f"segment_{i + 1:02d}")
        next_eval.insert(2, "evaluation_type", "next_segment_using_previous_params")
        next_eval_tables.append(next_eval)

    previous_params = params

current_segment_states = pd.concat(current_state_tables, ignore_index=True)
next_segment_evaluation = pd.concat(next_eval_tables, ignore_index=True) if next_eval_tables else pd.DataFrame()
segment_summary = pd.DataFrame(segment_rows)
segment_parameter_trace = pd.DataFrame(parameter_rows)


display(segment_summary)
display(segment_parameter_trace)




,segment_id,init_source,start,end,months,latest_date,latest_prob_high_state,latest_state,latest_regime_label,state_switches,avg_abs_prob_high_change
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,60,2010-11-30,0.071257,低均值状态,工业增加值低位区制,3,0.066033
1,segment_02,segment_01_params,2010-12-31,2015-11-30,60,2015-11-30,0.008337,低均值状态,工业增加值低位区制,5,0.114129
2,segment_03,segment_02_params,2015-12-31,2020-11-30,60,2020-11-30,1.000000,高均值状态,工业增加值高位区制,2,0.033898
3,segment_04,segment_03_params,2020-12-31,2025-11-30,60,2025-11-30,0.920580,高均值状态,工业增加值高位区制,0,0.006425


,segment_id,init_source,start,end,mu_low,mu_high,sigma,sigma_for_filter,p_stay_low,p_switch_low_to_high,p_switch_high_to_low,p_stay_high,init_prob_low,init_prob_high,log_likelihood,filtered_log_likelihood,iterations
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,-2.740011,5.240576,3.793072,3.224111,9.181769e-01,0.081823,0.080000,0.920000,1.178952e-27,1.0,-172.026242,-173.649513,20
1,segment_02,segment_01_params,2010-12-31,2015-11-30,-1.151918,2.746082,2.176628,1.850134,9.200000e-01,0.080000,0.223364,0.776636,1.064289e-08,1.0,-142.463404,-143.944922,41
2,segment_03,segment_02_params,2015-12-31,2020-11-30,-33.417896,-1.505434,2.823473,2.399952,5.133323e-32,1.000000,0.080000,0.920000,2.595339e-101,1.0,-154.692862,-156.464217,179
3,segment_04,segment_03_params,2020-12-31,2025-11-30,-1.659432,-0.854080,9.719043,8.261187,5.867439e-46,1.000000,0.082509,0.917491,5.987953e-106,1.0,-221.594329,-223.363905,500


## 8. 记录结论并导出结果

这一段汇总全样本与分段模型的最新判断，生成结论表，并导出 CSV、TXT 和 Excel。重点看输出目录里的 `M0000545_conclusions.txt` 和 `M0000545_hamcycle_full_segment_regime.xlsx`。



In [8]:
latest_full = full_states.tail(1).copy()
latest_segment = segment_summary.tail(1).copy()

conclusion_rows = [
    {
        "item": "data_range",
        "conclusion": (
            f"{SERIES_CODE} 原始数据范围为 {raw_series.index.min().date()} 至 {raw_series.index.max().date()}，"
            f"Hamilton cycle 可用范围为 {ham_cycle.index.min().date()} 至 {ham_cycle.index.max().date()}，"
            f"样本数 {len(ham_cycle)}。"
        ),
    },
    {
        "item": "full_sample_latest",
        "conclusion": (
            f"全样本区制切换模型最新一期为 {latest_full['date'].iloc[0].date()}，"
            f"高均值状态概率 {latest_full['prob_high_state'].iloc[0]:.3f}，"
            f"判定为 {latest_full['regime_label'].iloc[0]}。"
        ),
    },
    {
        "item": "segment_latest",
        "conclusion": (
            f"分段递推模型最新段为 {latest_segment['segment_id'].iloc[0]} "
            f"({latest_segment['start'].iloc[0].date()} 至 {latest_segment['end'].iloc[0].date()})，"
            f"段末高均值状态概率 {latest_segment['latest_prob_high_state'].iloc[0]:.3f}，"
            f"判定为 {latest_segment['latest_regime_label'].iloc[0]}。"
        ),
    },
    {
        "item": "model_comparison",
        "conclusion": (
            "若全样本与分段模型结论一致，可认为当前区制判断较稳健；"
            "若二者不一致，优先关注分段递推结果，因为它更强调近期参数结构。"
        ),
    },
]
conclusion_table = pd.DataFrame(conclusion_rows)

ham_data.to_csv(OUT / "M0000545_hamilton_cycle_series.csv", encoding="utf-8-sig")
full_states.to_csv(OUT / "M0000545_full_sample_states.csv", index=False, encoding="utf-8-sig")
full_parameter_table.to_csv(OUT / "M0000545_full_sample_parameters.csv", index=False, encoding="utf-8-sig")
segment_summary.to_csv(OUT / "M0000545_segment_summary.csv", index=False, encoding="utf-8-sig")
segment_parameter_trace.to_csv(OUT / "M0000545_segment_parameter_trace.csv", index=False, encoding="utf-8-sig")
current_segment_states.to_csv(OUT / "M0000545_segment_states.csv", index=False, encoding="utf-8-sig")
next_segment_evaluation.to_csv(OUT / "M0000545_next_segment_evaluation.csv", index=False, encoding="utf-8-sig")
conclusion_table.to_csv(OUT / "M0000545_conclusions.csv", index=False, encoding="utf-8-sig")

conclusion_text = "\n".join(f"- {row['conclusion']}" for _, row in conclusion_table.iterrows())
(OUT / "M0000545_conclusions.txt").write_text(conclusion_text, encoding="utf-8")

excel_path = OUT / "M0000545_hamcycle_full_segment_regime.xlsx"
try:
    with pd.ExcelWriter(excel_path) as writer:
        ham_data.reset_index().to_excel(writer, sheet_name="ham_cycle_series", index=False)
        full_states.to_excel(writer, sheet_name="full_sample_states", index=False)
        full_parameter_table.to_excel(writer, sheet_name="full_sample_params", index=False)
        segment_summary.to_excel(writer, sheet_name="segment_summary", index=False)
        segment_parameter_trace.to_excel(writer, sheet_name="segment_params", index=False)
        current_segment_states.to_excel(writer, sheet_name="segment_states", index=False)
        next_segment_evaluation.to_excel(writer, sheet_name="next_segment_eval", index=False)
        conclusion_table.to_excel(writer, sheet_name="conclusions", index=False)
except Exception as exc:
    print(f"Excel 导出失败，仅保留 CSV/TXT：{exc}")

print(f"已导出到：{OUT}")
print(conclusion_text)

try:
    display(ham_data.tail(12))
    display(full_parameter_table)
    display(full_states.tail(12))
    display(segment_summary)
    display(segment_parameter_trace)
    display(conclusion_table)
except NameError:
    pass




已导出到：c:\Users\16492\Desktop\实习内容\finance_intership\M0000545_hamcycle_full_segment_output
- M0000545 原始数据范围为 2005-01-31 至 2026-03-31，Hamilton cycle 可用范围为 2005-12-31 至 2026-03-31，样本数 244。
- 全样本区制切换模型最新一期为 2026-03-31，高均值状态概率 0.022，判定为 工业增加值低位区制。
- 分段递推模型最新段为 segment_04 (2020-12-31 至 2025-11-30)，段末高均值状态概率 0.921，判定为 工业增加值高位区制。
- 若全样本与分段模型结论一致，可认为当前区制判断较稳健；若二者不一致，优先关注分段递推结果，因为它更强调近期参数结构。


,raw_value,hamilton_trend,hamilton_cycle
date,,,
2025-04-30,6.1,6.991780,-0.891780
2025-05-31,5.8,7.005542,-1.205542
2025-06-30,6.8,7.056152,-0.256152
2025-07-31,5.7,7.062592,-1.362592
2025-08-31,5.2,7.283679,-2.083679
2025-09-30,6.5,4.658889,1.841111
2025-10-31,4.9,8.922463,-4.022463
2025-11-30,4.8,10.089295,-5.289295
2025-12-31,5.2,7.795359,-2.595359


,model_scope,code,name,start,end,nobs,mu_low,mu_high,sigma,sigma_for_filter,p_stay_low,p_switch_low_to_high,p_switch_high_to_low,p_stay_high,init_prob_low,init_prob_high,log_likelihood,filtered_log_likelihood,iterations
0,full_sample,M0000545,中国:工业增加值,2005-12-31,2026-03-31,244,-1.948174,6.320856,5.410018,4.598515,0.92,0.08,0.132743,0.867257,1.039761e-26,1.0,-782.736879,-788.928583,30


,model_scope,code,name,date,model_value,prior_prob_low_state,prior_prob_high_state,prob_low_state,prob_high_state,state,regime_label
232,full_sample,M0000545,中国:工业增加值,2025-04-30,-0.891780,0.317181,0.682819,0.607522,0.392478,低均值状态,工业增加值低位区制
233,full_sample,M0000545,中国:工业增加值,2025-05-31,-1.205542,0.611019,0.388981,0.855444,0.144556,低均值状态,工业增加值低位区制
234,full_sample,M0000545,中国:工业增加值,2025-06-30,-0.256152,0.806197,0.193803,0.915336,0.084664,低均值状态,工业增加值低位区制
235,full_sample,M0000545,中国:工业增加值,2025-07-31,-1.362592,0.853347,0.146653,0.958864,0.041136,低均值状态,工业增加值低位区制
236,full_sample,M0000545,中国:工业增加值,2025-08-31,-2.083679,0.887616,0.112384,0.976714,0.023286,低均值状态,工业增加值低位区制
237,full_sample,M0000545,中国:工业增加值,2025-09-30,1.841111,0.901668,0.098332,0.913006,0.086994,低均值状态,工业增加值低位区制
238,full_sample,M0000545,中国:工业增加值,2025-10-31,-4.022463,0.851513,0.148487,0.984849,0.015151,低均值状态,工业增加值低位区制
239,full_sample,M0000545,中国:工业增加值,2025-11-30,-5.289295,0.908072,0.091928,0.994587,0.005413,低均值状态,工业增加值低位区制
240,full_sample,M0000545,中国:工业增加值,2025-12-31,-2.595359,0.915739,0.084261,0.986014,0.013986,低均值状态,工业增加值低位区制
241,full_sample,M0000545,中国:工业增加值,2026-01-31,-3.677725,0.908990,0.091010,0.989993,0.010007,低均值状态,工业增加值低位区制


,segment_id,init_source,start,end,months,latest_date,latest_prob_high_state,latest_state,latest_regime_label,state_switches,avg_abs_prob_high_change
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,60,2010-11-30,0.071257,低均值状态,工业增加值低位区制,3,0.066033
1,segment_02,segment_01_params,2010-12-31,2015-11-30,60,2015-11-30,0.008337,低均值状态,工业增加值低位区制,5,0.114129
2,segment_03,segment_02_params,2015-12-31,2020-11-30,60,2020-11-30,1.000000,高均值状态,工业增加值高位区制,2,0.033898
3,segment_04,segment_03_params,2020-12-31,2025-11-30,60,2025-11-30,0.920580,高均值状态,工业增加值高位区制,0,0.006425


,segment_id,init_source,start,end,mu_low,mu_high,sigma,sigma_for_filter,p_stay_low,p_switch_low_to_high,p_switch_high_to_low,p_stay_high,init_prob_low,init_prob_high,log_likelihood,filtered_log_likelihood,iterations
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,-2.740011,5.240576,3.793072,3.224111,9.181769e-01,0.081823,0.080000,0.920000,1.178952e-27,1.0,-172.026242,-173.649513,20
1,segment_02,segment_01_params,2010-12-31,2015-11-30,-1.151918,2.746082,2.176628,1.850134,9.200000e-01,0.080000,0.223364,0.776636,1.064289e-08,1.0,-142.463404,-143.944922,41
2,segment_03,segment_02_params,2015-12-31,2020-11-30,-33.417896,-1.505434,2.823473,2.399952,5.133323e-32,1.000000,0.080000,0.920000,2.595339e-101,1.0,-154.692862,-156.464217,179
3,segment_04,segment_03_params,2020-12-31,2025-11-30,-1.659432,-0.854080,9.719043,8.261187,5.867439e-46,1.000000,0.082509,0.917491,5.987953e-106,1.0,-221.594329,-223.363905,500


,item,conclusion
0,data_range,M0000545 原始数据范围为 2005-01-31 至 2026-03-31，Hamil...
1,full_sample_latest,全样本区制切换模型最新一期为 2026-03-31，高均值状态概率 0.022，判定为 工业...
2,segment_latest,分段递推模型最新段为 segment_04 (2020-12-31 至 2025-11-30...
3,model_comparison,若全样本与分段模型结论一致，可认为当前区制判断较稳健；若二者不一致，优先关注分段递推结果，因...


## 输出文件说明

运行后重点查看：

- `M0000545_conclusions.txt`：文字结论
- `M0000545_conclusions.csv`：结构化结论
- `M0000545_hamcycle_full_segment_regime.xlsx`：全量结果表
- `M0000545_full_sample_states.csv`：全样本区制概率
- `M0000545_segment_summary.csv`：分段递推摘要

